# Phần 2: Ứng dụng Data Fitting vào dữ liệu thực tế

## Bài toán: Dự đoán giá nhà Melbourne bằng hồi quy tuyến tính

Notebook này sử dụng dữ liệu `Melbourne_housing_FULL.csv` để dự đoán biến mục tiêu `Price`. Quy trình gồm: EDA, xử lý missing values, feature engineering, pipeline tiền xử lý, huấn luyện OLS/Ridge bằng code tự cài từ Phần 1, đánh giá mô hình, phân tích phần dư và feature importance.

## 0. Cấu trúc thư mục cần dùng

Đặt file theo cấu trúc sau:

```text
1. ĐỒ ÁN/
├── part1/
│   ├── cross_validation.py
│   ├── linalg_utils.py
│   ├── ols_implementation.py
│   ├── residual_analysis.py
│   └── ridge_lasso.py
├── part2/
│   ├── data/
│   │   └── Melbourne_housing_FULL.csv
│   ├── data_pipeline.py
│   └── part2_notebook.ipynb
└── requirements.txt
```

Notebook này phải nằm ở `part2/part2_notebook.ipynb`, không nằm trong `part2/data/`.

In [ ]:
import os
import sys
import random
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Current working directory:")
print(os.getcwd())
print("\nFiles here:")
print(os.listdir("."))

## 1. Import code từ Phần 1 và `DataPipeline`

Phần 2 không sửa file code của Phần 1. Notebook chỉ import lại các hàm đã tự cài trong `part1/`, sau đó bổ sung một số hàm phụ trợ ngay trong notebook để chạy ổn hơn với dữ liệu thật.

In [ ]:
# Vì notebook nằm trong part2, còn code Phần 1 nằm trong ../part1
sys.path.append(os.path.abspath("../part1"))

# Import DataPipeline trong cùng thư mục part2
from data_pipeline import DataPipeline

# Import các hàm từ Phần 1
from ols_implementation import ols_fit, model_metrics, coef_inference, vif
from ridge_lasso import ridge_fit, ridge_trace
from cross_validation import _shuffle_indices, _split_into_folds
from residual_analysis import residual_plots
from linalg_utils import (
    matvec, transpose, matmul, inv,
    shape, mat_add, mat_scale, eye
)

print("Import code phần 1 và DataPipeline thành công!")

# 2. Đọc và mô tả dữ liệu

Dữ liệu được đọc từ file `data/Melbourne_housing_FULL.csv` vì notebook nằm trong thư mục `part2`.

In [ ]:
DATA_PATH = "data/Melbourne_housing_FULL.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Không tìm thấy file data. Hãy kiểm tra notebook có nằm trong thư mục part2 không "
        "và file CSV có nằm trong part2/data không."
    )

df = pd.read_csv(DATA_PATH)
print("Data shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

### Nhận xét ban đầu

Bộ dữ liệu có cả biến định lượng và biến phân loại. Biến mục tiêu là `Price`. Các biến định lượng như `Rooms`, `Distance`, `Bathroom`, `Landsize`, `BuildingArea`, `YearBuilt` có thể được đưa vào mô hình sau khi xử lý missing values và chuẩn hóa. Các biến phân loại như `Type`, `Method`, `Regionname`, `CouncilArea` cần được one-hot encoding.

# 3. Kiểm tra dữ liệu trùng và missing values

In [ ]:
duplicate_count = df.duplicated().sum()
print("Number of duplicated rows:", duplicate_count)

In [ ]:
df = df.drop_duplicates().copy()
print("Shape after dropping duplicates:", df.shape)

In [ ]:
missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_rate": df.isna().mean()
}).sort_values("missing_rate", ascending=False)

missing_table[missing_table["missing_count"] > 0]

In [ ]:
plt.figure(figsize=(10, 5))
missing_table[missing_table["missing_count"] > 0]["missing_rate"].plot(kind="bar")
plt.title("Missing Values Rate by Column")
plt.xlabel("Column")
plt.ylabel("Missing Rate")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Nhận xét missing values

Dữ liệu có nhiều cột bị thiếu giá trị. Vì `Price` là biến mục tiêu, các dòng thiếu `Price` sẽ bị loại bỏ. Với các biến giải thích, nhóm không xóa toàn bộ dòng thiếu vì có thể làm mất nhiều dữ liệu; thay vào đó, missing values sẽ được xử lý trong `DataPipeline` bằng median cho biến số và `Unknown` cho biến phân loại.

# 4. Xử lý biến mục tiêu `Price`

In [ ]:
df = df.dropna(subset=["Price"]).copy()
print("Shape after dropping missing Price:", df.shape)

In [ ]:
df["LogPrice"] = np.log1p(df["Price"])

print("Skewness of Price:", df["Price"].skew())
print("Skewness of LogPrice:", df["LogPrice"].skew())

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["Price"], bins=50)
plt.title("Distribution of Price")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(df["LogPrice"], bins=50)
plt.title("Distribution of log1p(Price)")
plt.xlabel("log1p(Price)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### Nhận xét biến mục tiêu

`Price` thường lệch phải do một số bất động sản có giá rất cao. Sau khi dùng `log1p(Price)`, phân phối cân đối hơn. Vì vậy, nhóm sử dụng `LogPrice` làm biến mục tiêu khi huấn luyện mô hình, rồi chuyển ngược về giá thật khi đánh giá MAE và RMSE.

# 5. Thống kê mô tả

In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

desc = df[numeric_cols].describe().T
desc["median"] = df[numeric_cols].median()
desc = desc[["count", "mean", "median", "std", "min", "25%", "50%", "75%", "max"]]

desc

### Nhận xét thống kê mô tả

Một số biến như `Landsize`, `BuildingArea` và `YearBuilt` có giá trị cực trị. Điều này gợi ý cần kiểm tra outliers và cân nhắc biến đổi log cho các biến diện tích.

# 6. Phân tích biến phân loại

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

cat_summary = pd.DataFrame({
    "unique_count": df[categorical_cols].nunique(dropna=False),
    "missing_count": df[categorical_cols].isna().sum(),
    "missing_rate": df[categorical_cols].isna().mean()
}).sort_values("unique_count", ascending=False)

cat_summary

In [ ]:
for col in ["Type", "Method", "Regionname", "CouncilArea"]:
    print("=" * 60)
    print(f"Column: {col}")
    print(df[col].value_counts(dropna=False).head(15))

In [ ]:
for col in ["Type", "Method", "Regionname"]:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=col, y="LogPrice")
    plt.title(f"log1p(Price) by {col}")
    plt.xlabel(col)
    plt.ylabel("log1p(Price)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

### Nhận xét biến phân loại

Các biến `Type`, `Method`, `Regionname`, `CouncilArea` có số lượng nhóm không quá lớn nên có thể one-hot encoding. Các biến như `Address`, `SellerG` có nhiều giá trị riêng biệt nên không đưa vào mô hình cơ bản để tránh làm tăng số chiều quá mức.

# 7. Phân phối các biến định lượng

In [ ]:
important_numeric = [
    "Rooms", "Distance", "Bedroom2", "Bathroom", "Car",
    "Landsize", "BuildingArea", "YearBuilt", "Propertycount"
]

for col in important_numeric:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col].dropna(), bins=50)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in ["Landsize", "BuildingArea"]:
    plt.figure(figsize=(7, 4))
    plt.hist(np.log1p(df[col].dropna()), bins=50)
    plt.title(f"Distribution of log1p({col})")
    plt.xlabel(f"log1p({col})")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

### Nhận xét phân phối biến số

`Landsize` và `BuildingArea` có phân phối lệch phải mạnh, nên nhóm tạo thêm `LogLandsize` và `LogBuildingArea` để giảm ảnh hưởng của outliers.

# 8. Phát hiện outliers bằng boxplot và IQR

In [ ]:
outlier_check_cols = [
    "Price", "Rooms", "Distance", "Bathroom", "Car",
    "Landsize", "BuildingArea", "YearBuilt", "Propertycount"
]

for col in outlier_check_cols:
    plt.figure(figsize=(8, 3))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
def iqr_outlier_summary(data, columns):
    rows = []
    for col in columns:
        s = data[col].dropna()
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = ((s < lower) | (s > upper)).sum()
        outlier_rate = outlier_count / len(s)
        rows.append({
            "column": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": outlier_count,
            "outlier_rate": outlier_rate
        })
    return pd.DataFrame(rows).sort_values("outlier_rate", ascending=False)

outlier_summary = iqr_outlier_summary(df, outlier_check_cols)
outlier_summary

### Nhận xét outliers

Một số biến có outliers, nhưng trong bối cảnh giá nhà, outliers không nhất thiết là lỗi dữ liệu. Nhóm không xóa toàn bộ outliers một cách cơ học, mà ưu tiên log-transform các biến lệch mạnh.

# 9. Phân tích tương quan

In [ ]:
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Numeric Variables")
plt.tight_layout()
plt.show()

In [ ]:
price_corr = corr_matrix["Price"].sort_values(key=lambda x: x.abs(), ascending=False)
price_corr

### Nhận xét tương quan

Các biến như `Rooms`, `Bedroom2`, `Bathroom` thường có tương quan dương với giá nhà. `Distance` thường có tương quan âm, phù hợp với trực giác rằng nhà càng xa trung tâm thì giá có xu hướng thấp hơn. Một số biến có thể mang thông tin trùng lặp, nên Ridge Regression được dùng để ổn định mô hình khi có đa cộng tuyến.

# 10. Feature engineering và chọn biến

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

df["SaleYear"] = df["Date"].dt.year
df["SaleMonth"] = df["Date"].dt.month
df["HouseAge"] = df["SaleYear"] - df["YearBuilt"]

df["LogLandsize"] = np.log1p(df["Landsize"])
df["LogBuildingArea"] = np.log1p(df["BuildingArea"])

df[["Date", "SaleYear", "SaleMonth", "YearBuilt", "HouseAge", "Landsize", "LogLandsize", "BuildingArea", "LogBuildingArea"]].head()

In [ ]:
numeric_features = [
    "Rooms", "Distance", "Bedroom2", "Bathroom", "Car",
    "LogLandsize", "LogBuildingArea", "HouseAge",
    "Lattitude", "Longtitude", "Propertycount"
]

categorical_features = [
    "Type", "Method", "Regionname", "CouncilArea"
]

X = df[numeric_features + categorical_features].copy()
y = df["LogPrice"].copy()

print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))
print("X shape:", X.shape)
print("y shape:", y.shape)

# 11. Train/test split

Dữ liệu được chia train/test trước khi fit pipeline để tránh data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train_raw.shape)
print("Test shape:", X_test_raw.shape)

# 12. Pipeline tiền xử lý

Notebook sử dụng `DataPipeline` trong file `data_pipeline.py`. Pipeline thực hiện xác định biến số/biến phân loại, điền missing values, one-hot encoding, căn chỉnh cột train/test và chuẩn hóa đặc trưng bằng mean/std học từ train set.

In [ ]:
pipeline = DataPipeline()

X_train_processed = pipeline.fit_transform(X_train_raw)
X_test_processed = pipeline.transform(X_test_raw)

# Thêm intercept sau khi pipeline đã chuẩn hóa xong.
# Không thêm intercept trước khi scale vì cột intercept không nên bị chuẩn hóa.
X_train_df = X_train_processed.copy()
X_test_df = X_test_processed.copy()

X_train_df.insert(0, "Intercept", 1.0)
X_test_df.insert(0, "Intercept", 1.0)

feature_names = X_train_df.columns.tolist()

X_train_list = X_train_df.values.tolist()
X_test_list = X_test_df.values.tolist()

y_train_list = y_train.tolist()
y_test_list = y_test.tolist()

print("Processed X_train shape:", X_train_df.shape)
print("Processed X_test shape:", X_test_df.shape)
print("Number of features after encoding:", len(feature_names))

### Nhận xét pipeline

Sau one-hot encoding, số lượng biến tăng lên so với dữ liệu ban đầu. Các biến đã được chuẩn hóa để hỗ trợ hồi quy Ridge và giúp so sánh hệ số hồi quy thuận tiện hơn. Cột `Intercept` được thêm sau cùng để phù hợp với công thức OLS tự cài.

# 13. Các hàm phụ trợ cho Phần 2

Các hàm dưới đây được viết ngay trong notebook để không phải sửa file code Phần 1. Chúng vẫn dùng các hàm ma trận tự cài trong `linalg_utils.py`.

In [ ]:
def evaluate_price(y_true_log, y_pred_log):
    # Đánh giá trên đơn vị giá thật bằng cách chuyển ngược expm1.
    y_true = np.expm1(np.array(y_true_log))
    y_pred = np.expm1(np.array(y_pred_log))
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
    return mae, rmse, r2


def mse_log(y_true_log, y_pred_log):
    y_true_log = np.array(y_true_log)
    y_pred_log = np.array(y_pred_log)
    return np.mean((y_true_log - y_pred_log) ** 2)


def ols_fit_stable(X, y):
    # OLS dùng lại các hàm ma trận từ Phần 1 nhưng không sửa file Phần 1.
    Xt = transpose(X)
    XtX = matmul(Xt, X)
    Xty = matvec(Xt, y)
    beta_hat = matvec(inv(XtX), Xty)
    n, p = len(X), len(X[0])
    y_hat = matvec(X, beta_hat)
    rss = sum((y[i] - y_hat[i]) ** 2 for i in range(n))
    sigma2 = rss / (n - p)
    return beta_hat, sigma2


def ridge_fit_no_intercept_penalty(X, y, lam):
    # Ridge Regression không regularize hệ số intercept.
    assert lam >= 0, "lambda must be non-negative"
    n, p = shape(X)
    Xt = transpose(X)
    XtX = matmul(Xt, X)
    Xty = matvec(Xt, y)
    I = eye(p)
    I[0][0] = 0.0
    XtX_reg = mat_add(XtX, mat_scale(I, lam))
    beta_hat = matvec(inv(XtX_reg), Xty)
    return beta_hat


def ridge_kfold_cv(X, y, lambdas, k=5, seed=42):
    # Chọn lambda cho Ridge bằng k-fold cross-validation.
    n = len(X)
    idx = _shuffle_indices(n, seed)
    folds = _split_into_folds(idx, k)
    results = []
    for lam in lambdas:
        fold_mses = []
        for j in range(k):
            val_idx = folds[j]
            train_idx = [i for jj in range(k) if jj != j for i in folds[jj]]
            X_train_fold = [X[i] for i in train_idx]
            y_train_fold = [y[i] for i in train_idx]
            X_val_fold = [X[i] for i in val_idx]
            y_val_fold = [y[i] for i in val_idx]
            beta = ridge_fit_no_intercept_penalty(X_train_fold, y_train_fold, lam)
            y_pred = matvec(X_val_fold, beta)
            fold_mses.append(mse_log(y_val_fold, y_pred))
        results.append({"lambda": lam, "cv_mse": sum(fold_mses) / k})
    results_df = pd.DataFrame(results)
    best = results_df.loc[results_df["cv_mse"].idxmin()].to_dict()
    return best, results_df


def make_cv_sample(X, y, max_n=3000, seed=42):
    # Lấy mẫu train để chạy Ridge CV nhanh hơn.
    n = len(X)
    if n <= max_n:
        return X, y
    rng = random.Random(seed)
    idx = rng.sample(range(n), max_n)
    X_sample = [X[i] for i in idx]
    y_sample = [y[i] for i in idx]
    return X_sample, y_sample

# 14. Mô hình 1: OLS Full

Mô hình OLS Full sử dụng toàn bộ biến sau tiền xử lý. Đây là mô hình baseline.

In [ ]:
model_outputs = {}

try:
    beta_ols_full, sigma2_full = ols_fit_stable(X_train_list, y_train_list)
    y_pred_full_log = matvec(X_test_list, beta_ols_full)
    mae_full, rmse_full, r2_full = evaluate_price(y_test_list, y_pred_full_log)
    model_outputs["OLS Full"] = {
        "beta": beta_ols_full,
        "features": feature_names,
        "X_test": X_test_list,
        "y_pred_log": y_pred_full_log,
        "MAE": mae_full,
        "RMSE": rmse_full,
        "R2": r2_full
    }
    print("OLS Full")
    print("MAE:", mae_full)
    print("RMSE:", rmse_full)
    print("R2:", r2_full)
except Exception as e:
    print("OLS Full không chạy được do lỗi:", repr(e))
    mae_full, rmse_full, r2_full = np.nan, np.nan, np.nan

# 15. Mô hình 2: OLS Selected

Mô hình OLS Selected sử dụng tập biến ít hơn để giảm đa cộng tuyến và tăng khả năng diễn giải.

In [ ]:
selected_features = [
    "Intercept",
    "Rooms", "Distance", "Bathroom", "Car",
    "LogLandsize", "LogBuildingArea", "HouseAge",
    "Lattitude", "Longtitude", "Propertycount"
]
selected_features = [col for col in selected_features if col in X_train_df.columns]
X_train_selected = X_train_df[selected_features].values.tolist()
X_test_selected = X_test_df[selected_features].values.tolist()
print("Selected feature count:", len(selected_features))
selected_features

In [ ]:
beta_ols_selected, sigma2_selected = ols_fit_stable(X_train_selected, y_train_list)
y_pred_selected_log = matvec(X_test_selected, beta_ols_selected)
mae_selected, rmse_selected, r2_selected = evaluate_price(y_test_list, y_pred_selected_log)
model_outputs["OLS Selected"] = {
    "beta": beta_ols_selected,
    "features": selected_features,
    "X_test": X_test_selected,
    "y_pred_log": y_pred_selected_log,
    "MAE": mae_selected,
    "RMSE": rmse_selected,
    "R2": r2_selected
}
print("OLS Selected")
print("MAE:", mae_selected)
print("RMSE:", rmse_selected)
print("R2:", r2_selected)

# 16. Mô hình 3: Ridge Regression

Ridge Regression được dùng để giảm ảnh hưởng của đa cộng tuyến và ổn định hệ số hồi quy. Lambda được chọn bằng 5-fold cross-validation trên một mẫu đại diện của train set để giảm thời gian chạy với code tự cài bằng Python thuần.

In [ ]:
X_train_cv, y_train_cv = make_cv_sample(
    X_train_list,
    y_train_list,
    max_n=3000,
    seed=RANDOM_STATE
)
print("Full train size:", len(X_train_list))
print("CV sample size:", len(X_train_cv))

lambdas = [0.01, 0.1, 1, 10, 100]

best_ridge, ridge_cv_results = ridge_kfold_cv(
    X_train_cv,
    y_train_cv,
    lambdas=lambdas,
    k=5,
    seed=RANDOM_STATE
)

best_lambda = best_ridge["lambda"]
print("Best lambda:", best_lambda)
ridge_cv_results

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(ridge_cv_results["lambda"], ridge_cv_results["cv_mse"], marker="o")
plt.xscale("log")
plt.title("Ridge Cross-Validation Curve")
plt.xlabel("lambda")
plt.ylabel("CV MSE on log(Price)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
beta_ridge = ridge_fit_no_intercept_penalty(X_train_list, y_train_list, best_lambda)
y_pred_ridge_log = matvec(X_test_list, beta_ridge)
mae_ridge, rmse_ridge, r2_ridge = evaluate_price(y_test_list, y_pred_ridge_log)
model_outputs["Ridge"] = {
    "beta": beta_ridge,
    "features": feature_names,
    "X_test": X_test_list,
    "y_pred_log": y_pred_ridge_log,
    "MAE": mae_ridge,
    "RMSE": rmse_ridge,
    "R2": r2_ridge
}
print("Ridge Regression")
print("Best lambda:", best_lambda)
print("MAE:", mae_ridge)
print("RMSE:", rmse_ridge)
print("R2:", r2_ridge)

# 17. So sánh mô hình

In [ ]:
results_df = pd.DataFrame([
    {"Model": "OLS Full", "MAE": mae_full, "RMSE": rmse_full, "R2": r2_full},
    {"Model": "OLS Selected", "MAE": mae_selected, "RMSE": rmse_selected, "R2": r2_selected},
    {"Model": "Ridge", "MAE": mae_ridge, "RMSE": rmse_ridge, "R2": r2_ridge},
])
results_df.sort_values("RMSE")

### Nhận xét so sánh mô hình

Mô hình có RMSE thấp nhất và R² cao nhất trên test set được xem là mô hình có khả năng dự đoán tốt nhất. Nếu Ridge cho kết quả tốt hơn OLS, điều đó cho thấy regularization giúp mô hình ổn định hơn khi dữ liệu có nhiều biến và có khả năng đa cộng tuyến.

# 18. Phân tích phần dư

In [ ]:
valid_results = results_df.dropna(subset=["RMSE"]).copy()
best_model_name = valid_results.sort_values("RMSE").iloc[0]["Model"]
best_info = model_outputs[best_model_name]
best_beta = best_info["beta"]
best_feature_names = best_info["features"]
best_y_pred_log = best_info["y_pred_log"]
print("Best model:", best_model_name)

In [ ]:
residuals_log = np.array(y_test_list) - np.array(best_y_pred_log)
plt.figure(figsize=(7, 5))
plt.scatter(best_y_pred_log, residuals_log, alpha=0.4, s=12)
plt.axhline(0, linestyle="--")
plt.title(f"Residuals vs Fitted - {best_model_name}")
plt.xlabel("Fitted log(Price)")
plt.ylabel("Residuals")
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats
plt.figure(figsize=(6, 6))
stats.probplot(residuals_log, dist="norm", plot=plt)
plt.title(f"Normal Q-Q Plot - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
sqrt_abs_resid = np.sqrt(np.abs(residuals_log))
plt.figure(figsize=(7, 5))
plt.scatter(best_y_pred_log, sqrt_abs_resid, alpha=0.4, s=12)
plt.title(f"Scale-Location Plot - {best_model_name}")
plt.xlabel("Fitted log(Price)")
plt.ylabel("sqrt(|Residuals|)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(residuals_log, bins=50)
plt.title(f"Distribution of Residuals - {best_model_name}")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### Nhận xét phần dư

Các biểu đồ phần dư giúp kiểm tra mức độ phù hợp của hồi quy tuyến tính. Nếu phần dư phân tán ngẫu nhiên quanh 0, mô hình phù hợp tương đối tốt. Nếu có dạng hình phễu hoặc lệch mạnh ở hai đuôi Q-Q plot, mô hình tuyến tính vẫn còn hạn chế do outliers hoặc quan hệ phi tuyến trong dữ liệu giá nhà.

# 19. Feature importance

In [ ]:
coef_df = pd.DataFrame({
    "feature": best_feature_names,
    "coef": best_beta
})
coef_df = coef_df[coef_df["feature"] != "Intercept"].copy()
coef_df["abs_coef"] = coef_df["coef"].abs()
top_coef = coef_df.sort_values("abs_coef", ascending=False).head(15)
top_coef

In [ ]:
plt.figure(figsize=(8, 6))
plt.barh(top_coef["feature"], top_coef["coef"])
plt.title(f"Top 15 Feature Coefficients - {best_model_name}")
plt.xlabel("Coefficient")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Nhận xét feature importance

Các biến có hệ số dương làm tăng `LogPrice`, trong khi các biến có hệ số âm làm giảm `LogPrice`. Những biến liên quan đến quy mô nhà, tiện ích, vị trí và khu vực thường có ảnh hưởng đáng kể đến giá. Nếu `Distance` có hệ số âm, điều này phù hợp với trực giác rằng nhà càng xa trung tâm thì giá có xu hướng thấp hơn.

# 20. Kết luận

Qua quá trình thực nghiệm, nhóm rút ra các kết luận chính:

1. Dữ liệu Melbourne Housing có missing values đáng kể ở nhiều biến, đặc biệt là các biến diện tích, năm xây dựng và thông tin phòng/xe.
2. Biến mục tiêu `Price` có phân phối lệch phải, do đó sử dụng `log1p(Price)` giúp hồi quy tuyến tính ổn định hơn.
3. Các biến về số phòng, khoảng cách đến trung tâm, diện tích, vị trí và khu vực có liên hệ đáng kể với giá nhà.
4. Trong ba mô hình OLS Full, OLS Selected và Ridge Regression, mô hình có RMSE thấp nhất trên test set được chọn làm mô hình tốt nhất.
5. Hạn chế của mô hình tuyến tính là khó nắm bắt hoàn toàn quan hệ phi tuyến và outliers trong dữ liệu giá nhà. Hướng mở rộng có thể là Polynomial Regression, Lasso hoặc Kernel Ridge Regression.

In [ ]:
results_df.to_csv("model_comparison_results.csv", index=False)
top_coef.to_csv("top_feature_coefficients.csv", index=False)
print("Saved model_comparison_results.csv")
print("Saved top_feature_coefficients.csv")